In [1]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

In [2]:
df = pd.read_csv(
    "/home1/smaruj/akitaV2-analyses/input_data/preprocess_boundary_CTCFs/output/CTCFs_jaspar_filtered_mm10.tsv", 
    sep="\t"
)

In [3]:
from pyfaidx import Fasta

In [4]:
genome = Fasta('/project2/fudenber_735/genomes/mm10/mm10.fa')

In [5]:
def calculate_gc_content(sequence):
    """Calculate GC content from a DNA sequence string."""
    sequence = sequence.upper()
    gc_count = sequence.count('G') + sequence.count('C')
    total = len(sequence)
    return gc_count / total if total > 0 else 0.0

In [6]:
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing natural CTCFs"):
    chrom = row['chrom']
    start = row['start']
    end = row['end']
    
    # Extract sequence with 30bp flanks
    ext_start = start - 30
    ext_end = end + 30
    
    try:
        seq = str(genome[chrom][ext_start:ext_end]).upper()
    except:
        print(f"Warning: Could not extract sequence for {chrom}:{ext_start}-{ext_end}")
        continue
    
    # If there's a strand column, reverse complement if needed
    if 'strand' in row and row['strand'] == '-':
        complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C', 'N': 'N'}
        seq = ''.join(complement.get(base, base) for base in reversed(seq))
    
    # Core is in the middle (with 30bp flanks on each side)
    core_length = end - start
    core_start_idx = 30
    core_end_idx = 30 + core_length
    
    df.at[idx, 'gc_core'] = calculate_gc_content(seq[core_start_idx:core_end_idx])
    df.at[idx, 'gc_upstream_15'] = calculate_gc_content(seq[core_start_idx-15:core_start_idx])
    df.at[idx, 'gc_upstream_30'] = calculate_gc_content(seq[core_start_idx-30:core_start_idx])
    df.at[idx, 'gc_downstream_15'] = calculate_gc_content(seq[core_end_idx:core_end_idx+15])
    df.at[idx, 'gc_downstream_30'] = calculate_gc_content(seq[core_end_idx:core_end_idx+30])
    df.at[idx, 'gc_core_plus_15'] = calculate_gc_content(seq[core_start_idx-15:core_end_idx+15])
    df.at[idx, 'gc_core_plus_30'] = calculate_gc_content(seq[core_start_idx-30:core_end_idx+30])

print("\nGC content calculated for natural CTCFs!")
print(df[['gc_core', 'gc_core_plus_15', 'gc_core_plus_30', 
                              'gc_upstream_15', 'gc_downstream_15']].describe())

Processing natural CTCFs: 100%|██████████| 7560/7560 [00:03<00:00, 2115.84it/s]


GC content calculated for natural CTCFs!
           gc_core  gc_core_plus_15  gc_core_plus_30  gc_upstream_15  \
count  7560.000000      7560.000000      7560.000000     7560.000000   
mean      0.598246         0.536481         0.519036        0.494647   
std       0.097682         0.092004         0.088392        0.140315   
min       0.263158         0.224490         0.240506        0.066667   
25%       0.526316         0.469388         0.455696        0.400000   
50%       0.578947         0.530612         0.518987        0.466667   
75%       0.684211         0.591837         0.569620        0.600000   
max       0.947368         0.877551         0.860759        1.000000   

       gc_downstream_15  
count       7560.000000  
mean           0.500079  
std            0.143580  
min            0.000000  
25%            0.400000  
50%            0.533333  
75%            0.600000  
max            0.933333  


In [7]:
# Save
output_path = "/scratch1/smaruj/suppressing_CTCFs/results_repeated/natural_ctcfs_with_gc.tsv"
df.to_csv(output_path, sep="\t", index=False)
print(f"\nSaved to: {output_path}")


Saved to: /scratch1/smaruj/suppressing_CTCFs/results_repeated/natural_ctcfs_with_gc.tsv
